# NLP Practical Exam — Text Processing + Language Modeling (90 minutes)

**Instructions**
- Work in this notebook only.
- Write short, clear comments to justify *tool choices* (regex vs NLTK, etc.).
- Do **not** use external NLP libraries beyond **NLTK**, **NumPy**, **PyTorch** (PyTorch not needed here).
- Keep outputs readable (print key variables).

**Total: 10 points**


## Given text

```python
text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.")
```

> Treat the text as *synthetic exam data* (no fact-checking needed).


## Questions

1. **(1 pt)** Sentence splitting using **regex + NLTK**.
2. **(1 pt)** Regex normalization: acronyms, height meters→centimeters, money `$X.Y billion` → `x point y billion` (words).
3. **(1 pt)** Lowercase **except** proper nouns; join multiword proper nouns with underscore (e.g., `Sam Altman → Sam_Altman`). Keep acronyms uppercase.
4. **(1 pt)** Tokenize (tool of your choice).
5. **(1 pt)** Remove stopwords (tool of your choice); keep entity tokens.
6. **(1 pt)** Create bigrams with pure Python.
7. **(2 pt)** Build a bigram LM (MLE) and `predict_next(prev_word, top_k=3)`.

8. **(2 pt)** Implement a simple **BPE** on: `corpus = "low lower newest widest"` (≥5 merges or until no merges).
9. **(1 pt)** Compute Accuracy/Precision/Recall/F1 for an invented confusion matrix (explain with comments).


In [2]:
import re
import math
import nltk
from collections import Counter, defaultdict

# NLTK downloads (safe to run multiple times)
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

from nltk.tokenize import sent_tokenize, word_tokenize
from nltk.corpus import stopwords

text = ("In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. "
        "He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. "
        "A report valued the project at $3.2 billion.")

print(text)


In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O. A report valued the project at $3.2 billion.


## Q1

In [7]:
protected_text = re.sub(r'(?<=[A-Z])\.(?!\s+[A-Z])', '<DOT>', text)
sentences_raw = sent_tokenize(protected_text)

# Restore the dots so the acronyms are back to normal
sentences = [s.replace('<DOT>', '.') for s in sentences_raw]

# print(sentences)
print("--- Q1: SENTENCES ---")
for i, s in enumerate(sentences):
    print(f"{i+1}. {s}")


--- Q1: SENTENCES ---
1. In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona.
2. He is 1.86m tall and met with researchers from U.P.C. and U.N.E.S.C.O.
3. A report valued the project at $3.2 billion.


## Q2

In [8]:
# Q2 (1 pt): Regex normalization
text_norm = text

# 1. Acronyms: U.P.C. -> UPC. 
# Match uppercase letters followed by a dot, repeated 2+ times, and just strip the dots.
text_norm = re.sub(r'(?:[A-Z]\.){2,}', lambda m: m.group().replace('.', ''), text_norm)

# 2. Height: 1.86m -> 186 centimeters.
# Capturing the whole number and decimals, doing the math inside a lambda.
text_norm = re.sub(r'(\d)\.(\d{2})m', lambda m: f"{int(m.group(1))*100 + int(m.group(2))} centimeters", text_norm)

# 3. Money: $3.2 billion -> three point two billion.
# A quick dictionary to map digits to words. 
num_to_word = {'0':'zero', '1':'one', '2':'two', '3':'three', '4':'four', '5':'five', '6':'six', '7':'seven', '8':'eight', '9':'nine'}
text_norm = re.sub(r'\$(\d)\.(\d)\s+billion', lambda m: f"{num_to_word[m.group(1)]} point {num_to_word[m.group(2)]} billion", text_norm)

print("--- Q2: NORMALIZED TEXT ---")
print(text_norm, "\n")


--- Q2: NORMALIZED TEXT ---
In mid-February 2026, the CEO of OpenAI, Sam Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion. 



## Q3

In [9]:
# Q3 (1 pt): Lowercase except proper nouns + underscore multiword proper nouns
text_case = re.sub(r'\b([A-Z][a-z]+(?:\s+[A-Z][a-z]+)+)\b', lambda m: m.group().replace(' ', '_'), text_norm)

#I split the words
words = text_case.split()
processed_words = []

for w in words:
    clean_w = re.sub(r'[^A-Za-z0-9_]', '', w) # Strip punctuation just for the checks
    
    # Keep acronyms (all caps) uppercase
    if clean_w.isupper() and len(clean_w) > 1:
        processed_words.append(w)
    # Keep MixedCase as-is (OpenAI) -> checks if there are uppercase letters after the first char
    elif any(c.isupper() for c in clean_w[1:]):
        processed_words.append(w)
    # Keep underscored proper nouns as-is (Sam_Altman)
    elif '_' in clean_w:
        processed_words.append(w)
    # Proper nouns like 'Barcelona' (Capitalized word). 
    # Note: this will also catch sentence starters like 'In' and 'He', but without a POS tagger, this is the safest heuristic!
    elif clean_w.istitle():
        processed_words.append(w)
    # Lowercase everything else!
    else:
        processed_words.append(w.lower())

text_case = " ".join(processed_words)

print("--- Q3: CASED & UNDERSCORED ---")
print(text_case, "\n")

--- Q3: CASED & UNDERSCORED ---
In mid-February 2026, the CEO of OpenAI, Sam_Altman, visited Barcelona. He is 186 centimeters tall and met with researchers from UPC and UNESCO A report valued the project at three point two billion. 



## Q4

In [10]:
# Q4 (1 pt): Tokenization
# Sticking with NLTK's word_tokenize. It handles remaining punctuation better.
tokens = word_tokenize(text_case)

print("--- Q4: TOKENS ---")
print(tokens, "\n")


--- Q4: TOKENS ---
['In', 'mid-February', '2026', ',', 'the', 'CEO', 'of', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', 'He', 'is', '186', 'centimeters', 'tall', 'and', 'met', 'with', 'researchers', 'from', 'UPC', 'and', 'UNESCO', 'A', 'report', 'valued', 'the', 'project', 'at', 'three', 'point', 'two', 'billion', '.'] 



## Q5

In [11]:
# Q5 (1 pt): Stopword removal
stop_words = set(stopwords.words('english'))
# We keep the token if its lowercase version is NOT in stopwords, or if it's punctuation.
# This perfectly protects entities like "OpenAI" and "Sam_Altman" because they aren't in the stopwords list.
tokens_nostop = [t for t in tokens if t.lower() not in stop_words or not t.isalpha()]

print("--- Q5: TOKENS WITHOUT STOPWORDS ---")
print(tokens_nostop, "\n")


--- Q5: TOKENS WITHOUT STOPWORDS ---
['mid-February', '2026', ',', 'CEO', 'OpenAI', ',', 'Sam_Altman', ',', 'visited', 'Barcelona', '.', '186', 'centimeters', 'tall', 'met', 'researchers', 'UPC', 'UNESCO', 'report', 'valued', 'project', 'three', 'point', 'two', 'billion', '.'] 



## Q6

In [12]:
# Q6 (1 pt): Bigrams with pure Python (no NLTK bigrams helper)
# Justification: A simple list comprehension pairing the current token with the next one using an offset index.
bigrams = [(tokens_nostop[i], tokens_nostop[i+1]) for i in range(len(tokens_nostop)-1)]

print("--- Q6: BIGRAMS ---")
print(bigrams[:5], "... (truncated)\n")


--- Q6: BIGRAMS ---
[('mid-February', '2026'), ('2026', ','), (',', 'CEO'), ('CEO', 'OpenAI'), ('OpenAI', ',')] ... (truncated)



## Q7

In [13]:
# Q7 (2 pt): Bigram Language Model + next-word prediction
bigram_counts = Counter(bigrams)
# context_counts measures how many times w1 appears as the first word in a bigram
context_counts = Counter([b[0] for b in bigrams]) 
model = defaultdict(dict)

# MLE Formula: P(w2|w1) = count(w1, w2) / count(w1)
for (w1, w2), count in bigram_counts.items():
    model[w1][w2] = count / context_counts[w1]

def predict_next(prev_word, model, top_k=3):
    if prev_word not in model:
        return []
    # Sort predictions by probability (descending)
    next_words = sorted(model[prev_word].items(), key=lambda x: x[1], reverse=True)
    return next_words[:top_k]

print("--- Q7: PREDICT NEXT ('OpenAI') ---")
print(predict_next("OpenAI", model, top_k=3), "\n")


--- Q7: PREDICT NEXT ('OpenAI') ---
[(',', 1.0)] 



## Q8

In [ ]:
# Q8 (2 pt): Simple BPE (Byte Pair Encoding) on a tiny corpus
corpus = "low lower newest widest"

# Requirements:
# - Represent each word as characters + </w>
# - Compute pair frequencies (weighted by word frequency)
# - Merge most frequent pair
# - Do at least 5 merges (or stop if no pairs)
#
# Deliver:
# - merges: list of merges in order
# - final segmented version of each word

merges = None

# TODO: implement BPE helper functions:
# - get_vocab_from_corpus
# - get_pair_frequencies
# - merge_pair_in_vocab

# print(merges)


## Q9

In [ ]:
# Q9 (1 pt): Metrics — Accuracy, Precision, Recall, F1
# Invent a confusion matrix (TP, FP, FN, TN) and compute metrics.
# Explain each formula briefly in comments.

TP = None
FP = None
FN = None
TN = None

accuracy = None
precision = None
recall = None
f1 = None

# print(accuracy, precision, recall, f1)
